# Data Analysis with SQLite
## Olist Brazilian E-Commerce · Reverse Logistics Analysis

**Input:** `olist_analytical_table.csv` (output từ notebook EDA & Cleaning)

**Mục tiêu:** Trả lời 3 Research Questions bằng SQL queries thông qua SQLite

```
RQ1 — Delivery Performance & Customer Satisfaction
RQ2 — Product Characteristics & Risk of Low Ratings  
RQ3 — Geographic Factors & Logistics Efficiency
```

---

**Pipeline notebook này:**
```
1. Load CSV → nạp vào SQLite (in-memory database)
2. RQ1 Queries  → delivery time, delay, freight vs review score
3. RQ2 Queries  → product category & dimensions vs low ratings
4. RQ3 Queries  → interstate shipping vs delivery, freight, satisfaction
5. Export kết quả → CSV để dùng tiếp ở bước Visualization
```

## 0. Import thư viện & Kết nối SQLite

In [14]:
import sqlite3
import pandas as pd
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)

# ── Load analytical table ────────────────────────────────
CSV_PATH = "olist_analytical_table.csv"
df = pd.read_csv(CSV_PATH)
print(f"Đọc CSV thành công: {df.shape[0]:,} rows × {df.shape[1]} cols")

# ── Tạo SQLite in-memory database ───────────────────────
conn = sqlite3.connect(":memory:")

# Nạp toàn bộ DataFrame vào bảng SQL tên 'orders'
df.to_sql("orders", conn, index=False, if_exists="replace")
print("Đã nạp dữ liệu vào SQLite (bảng: orders)")

# Helper function: chạy query → trả về DataFrame
def run_query(sql, title=""):
    """Chạy SQL query và trả về DataFrame có tiêu đề."""
    result = pd.read_sql_query(sql, conn)
    if title:
        print(f"\n{'='*60}")
        print(f" {title}")
        print(f"{'='*60}")
    return result

print("\nCác cột trong bảng orders:")
print(df.columns.tolist())

Đọc CSV thành công: 99,441 rows × 49 cols
Đã nạp dữ liệu vào SQLite (bảng: orders)

Các cột trong bảng orders:
['order_id', 'customer_id', 'customer_unique_id', 'customer_city', 'customer_state', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'purchase_year', 'purchase_month', 'purchase_quarter', 'purchase_dow', 'actual_delivery_days', 'estimated_delivery_days', 'delay_days', 'is_late_delivery', 'total_items', 'total_price', 'total_freight', 'total_order_value', 'avg_item_price', 'unique_sellers', 'has_outlier_price', 'has_outlier_freight', 'total_payment_value', 'max_installments', 'payment_type_count', 'primary_payment_type', 'has_outlier_payment', 'avg_review_score', 'review_count', 'has_comment', 'has_review', 'product_id', 'product_category_name', 'product_category_name_english', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'selle

## 1. Kiểm tra bảng SQLite (Sanity Check)

In [15]:
# Xem cấu trúc bảng (tương đương df.info())
schema_query = """
    SELECT name AS column_name,
           type AS data_type
    FROM   pragma_table_info('orders')
    ORDER  BY cid
"""
schema = run_query(schema_query, "Cấu trúc bảng orders")
display(schema)

# Xem trước 3 dòng đầu
preview = run_query("SELECT * FROM orders LIMIT 3", "Xem trước dữ liệu (3 dòng)")
display(preview)

# Đếm tổng số đơn
count = run_query("SELECT COUNT(*) AS total_orders FROM orders", "Tổng số đơn hàng")
display(count)


 Cấu trúc bảng orders


,column_name,data_type
0,order_id,TEXT
1,customer_id,TEXT
2,customer_unique_id,TEXT
3,customer_city,TEXT
4,customer_state,TEXT
5,order_status,TEXT
6,order_purchase_timestamp,TEXT
7,order_approved_at,TEXT
8,order_delivered_carrier_date,TEXT
9,order_delivered_customer_date,TEXT



 Xem trước dữ liệu (3 dòng)


,order_id,customer_id,customer_unique_id,customer_city,customer_state,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_year,purchase_month,purchase_quarter,purchase_dow,actual_delivery_days,estimated_delivery_days,delay_days,is_late_delivery,total_items,total_price,total_freight,total_order_value,avg_item_price,unique_sellers,has_outlier_price,has_outlier_freight,total_payment_value,max_installments,payment_type_count,primary_payment_type,has_outlier_payment,avg_review_score,review_count,has_comment,has_review,product_id,product_category_name,product_category_name_english,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_id,seller_city,seller_state,same_state,is_delivered,delivery_efficiency
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,10,4,0,8.0000,15,-7.0000,0,1.0000,29.9900,8.7200,38.7100,29.9900,1.0000,0.0000,0.0000,43.4000,1.0000,2.0000,voucher,1.0000,4.0000,1.0000,1.0000,1,87285b34884572647811a353c7ac498a,utilidades_domesticas,housewares,500.0000,19.0000,8.0000,13.0000,3504c0cb71d7fa48d967e0e4c94d59d9,maua,SP,1,1,7.0000
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,barreiras,BA,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018,7,3,1,13.0000,19,-6.0000,0,1.0000,118.7000,22.7600,141.4600,118.7000,1.0000,0.0000,0.0000,141.4600,1.0000,1.0000,boleto,0.0000,4.0000,1.0000,1.0000,1,595fac2a385ac33a80bd5114aec74eb8,perfumaria,perfumery,400.0000,19.0000,13.0000,19.0000,289cdb325fb7e7f891c38608bf9e0962,belo horizonte,SP,0,1,6.0000
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018,8,3,2,9.0000,26,-17.0000,0,1.0000,159.9000,19.2200,179.1200,159.9000,1.0000,0.0000,0.0000,179.1200,3.0000,1.0000,credit_card,0.0000,5.0000,1.0000,0.0000,1,aa4383b373c6aca5d8797843e5594415,automotivo,auto,420.0000,24.0000,19.0000,21.0000,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,0,1,17.0000



 Tổng số đơn hàng


,total_orders
0,99441


---
## 2. RQ1 — Delivery Performance & Customer Satisfaction

> **RQ1:** To what extent do delivery-related factors, including actual delivery time, delivery delay, and freight cost, influence customer review scores?

**Chiến lược phân tích:**
- Nhóm đơn hàng theo `avg_review_score` để so sánh các chỉ số delivery
- Phân tích đơn trễ (`is_late_delivery = 1`) vs đúng hạn (`= 0`)
- Chia `total_freight` thành nhóm (bucket) và xem review score tương ứng

### RQ1 — Query 1: Chỉ số giao hàng trung bình theo từng mức Review Score

In [16]:
# Mục tiêu: xem delivery time, delay, freight thay đổi như thế nào
#           khi review score tăng từ 1 → 5
# Chỉ lấy đơn đã giao (order_status = 'delivered') để có đủ dữ liệu

rq1_q1 = """
    SELECT
        ROUND(avg_review_score) AS review_score,

        COUNT(*)                                        AS total_orders,

        -- Thời gian giao hàng thực tế (ngày)
        ROUND(AVG(actual_delivery_days),  1)            AS avg_delivery_days,

        -- Số ngày trễ so với dự kiến (>0 = trễ, <0 = sớm)
        ROUND(AVG(delay_days),            1)            AS avg_delay_days,

        -- Phí vận chuyển trung bình
        ROUND(AVG(total_freight),         2)            AS avg_freight_cost,

        -- Tỉ lệ đơn giao trễ trong nhóm này
        ROUND(
            100.0 * SUM(is_late_delivery) / COUNT(*), 1
        )                                               AS late_delivery_pct

    FROM  orders
    WHERE order_status       = 'delivered'
      AND avg_review_score   IS NOT NULL
      AND actual_delivery_days IS NOT NULL

    GROUP BY ROUND(avg_review_score)
    ORDER BY review_score
"""

rq1_result1 = run_query(rq1_q1,
    "RQ1-Q1: Chỉ số giao hàng trung bình theo mức Review Score (1–5)")
display(rq1_result1)

print("\n Diễn giải:")
print("  - Score càng thấp → avg_delivery_days và avg_delay_days có xu hướng cao hơn")
print("  - late_delivery_pct cao ở score 1–2 → giao hàng trễ ảnh hưởng review")


 RQ1-Q1: Chỉ số giao hàng trung bình theo mức Review Score (1–5)


,review_score,total_orders,avg_delivery_days,avg_delay_days,avg_freight_cost,late_delivery_pct
0,1.0000,9312,20.9000,-3.4000,27.5300,37.5000
1,2.0000,2924,16.2000,-8.0000,25.7900,19.9000
2,3.0000,7946,13.8000,-10.2000,23.2700,10.1000
3,4.0000,18892,11.8000,-11.8000,22.0200,4.4000
4,5.0000,56750,10.2000,-12.8000,21.3400,2.5000



 Diễn giải:
  - Score càng thấp → avg_delivery_days và avg_delay_days có xu hướng cao hơn
  - late_delivery_pct cao ở score 1–2 → giao hàng trễ ảnh hưởng review


### RQ1 — Query 2: Đơn trễ vs Đúng hạn — So sánh Review Score

In [17]:
# Mục tiêu: so sánh trực tiếp 2 nhóm (trễ / đúng hạn)
# để thấy tác động của delay đến review

rq1_q2 = """
    SELECT
        -- Nhãn dễ đọc cho 2 nhóm
        CASE is_late_delivery
            WHEN 1 THEN 'Giao trễ'
            WHEN 0 THEN 'Đúng hạn / Sớm'
            ELSE        'Không xác định'
        END                                             AS delivery_status,

        COUNT(*)                                        AS total_orders,

        -- Review score trung bình của từng nhóm
        ROUND(AVG(avg_review_score),      2)            AS avg_review_score,

        -- Tỉ lệ đơn bị review thấp (score ≤ 2)
        ROUND(
            100.0 * SUM(CASE WHEN avg_review_score <= 2 THEN 1 ELSE 0 END)
                  / COUNT(*), 1
        )                                               AS low_score_pct,

        -- Số ngày trễ trung bình
        ROUND(AVG(delay_days), 1)                       AS avg_delay_days,

        -- Thời gian giao thực tế trung bình
        ROUND(AVG(actual_delivery_days), 1)             AS avg_delivery_days

    FROM  orders
    WHERE order_status         = 'delivered'
      AND avg_review_score     IS NOT NULL
      AND is_late_delivery     IS NOT NULL

    GROUP BY is_late_delivery
    ORDER BY is_late_delivery DESC
"""

rq1_result2 = run_query(rq1_q2,
    "RQ1-Q2: So sánh Review Score giữa Đơn Trễ và Đúng Hạn")
display(rq1_result2)

print("\n Diễn giải:")
print("  - low_score_pct: % đơn bị đánh giá 1–2 sao trong mỗi nhóm")
print("  - Nhóm 'Giao trễ' dự kiến có avg_review_score thấp hơn đáng kể")


 RQ1-Q2: So sánh Review Score giữa Đơn Trễ và Đúng Hạn


,delivery_status,total_orders,avg_review_score,low_score_pct,avg_delay_days,avg_delivery_days
0,Giao trễ,7147,2.4600,57.0000,10.0000,31.9000
1,Đúng hạn / Sớm,88685,4.2900,9.2000,-13.0000,10.5000



 Diễn giải:
  - low_score_pct: % đơn bị đánh giá 1–2 sao trong mỗi nhóm
  - Nhóm 'Giao trễ' dự kiến có avg_review_score thấp hơn đáng kể


---
## 3. RQ2 — Product Characteristics & Risk of Low Ratings

> **RQ2:** How do product category and physical product characteristics (weight and dimensions) relate to the likelihood of receiving low customer ratings?

**Chiến lược phân tích:**
- Top/Bottom danh mục theo tỉ lệ đánh giá thấp (`avg_review_score ≤ 2`)
- So sánh cân nặng và thể tích sản phẩm giữa nhóm low-rating vs high-rating

### RQ2 — Query 3: Tỉ lệ Low Rating theo Danh Mục Sản Phẩm (Top 15)

In [18]:
# Mục tiêu: tìm danh mục nào có tỉ lệ đánh giá thấp (≤2 sao) cao nhất
# Low rating = avg_review_score ≤ 2 (thang 1–5)
# Chỉ lấy danh mục có ít nhất 100 đơn để tránh bias từ nhóm nhỏ

rq2_q3 = """
    SELECT
        -- Tên danh mục tiếng Anh (dễ đọc hơn)
        COALESCE(
            product_category_name_english,
            product_category_name,
            'Unknown'
        )                                               AS category,

        COUNT(*)                                        AS total_orders,

        -- Review score trung bình của danh mục
        ROUND(AVG(avg_review_score), 2)                 AS avg_review_score,

        -- Số đơn bị low rating (≤ 2 sao)
        SUM(
            CASE WHEN avg_review_score <= 2 THEN 1 ELSE 0 END
        )                                               AS low_rating_orders,

        -- Tỉ lệ % đơn bị low rating trong danh mục
        ROUND(
            100.0 * SUM(CASE WHEN avg_review_score <= 2 THEN 1 ELSE 0 END)
                  / COUNT(*), 1
        )                                               AS low_rating_pct

    FROM  orders
    WHERE avg_review_score IS NOT NULL
      AND product_category_name IS NOT NULL

    GROUP BY category

    -- Lọc danh mục có đủ lượng đơn để kết quả có ý nghĩa thống kê
    HAVING COUNT(*) >= 100

    ORDER BY low_rating_pct DESC
    LIMIT 15
"""

rq2_result3 = run_query(rq2_q3,
    "RQ2-Q3: Top 15 Danh Mục có Tỉ Lệ Low Rating (≤2 sao) Cao Nhất")
display(rq2_result3)

print("\n Diễn giải:")
print("  - Danh mục đầu bảng = có nguy cơ bị đánh giá xấu cao nhất")
print("  - Có thể liên quan đến sản phẩm dễ vỡ, cồng kềnh, hoặc dịch vụ kém")


 RQ2-Q3: Top 15 Danh Mục có Tỉ Lệ Low Rating (≤2 sao) Cao Nhất


,category,total_orders,avg_review_score,low_rating_orders,low_rating_pct
0,fashion_male_clothing,111,3.7000,29,26.1000
1,office_furniture,1255,3.6200,283,22.5000
2,audio,343,3.8400,74,21.6000
3,construction_tools_safety,161,3.8900,31,19.3000
4,unknown,1405,3.9300,270,19.2000
5,fixed_telephony,214,3.9000,40,18.7000
6,home_confort,373,3.8800,69,18.5000
7,fashion_underwear_beach,120,3.9300,22,18.3000
8,furniture_decor,6305,4.0200,1027,16.3000
9,bed_bath_table,9207,3.9800,1505,16.3000



 Diễn giải:
  - Danh mục đầu bảng = có nguy cơ bị đánh giá xấu cao nhất
  - Có thể liên quan đến sản phẩm dễ vỡ, cồng kềnh, hoặc dịch vụ kém


### RQ2 — Query 4: Đặc Điểm Vật Lý Sản Phẩm — Low Rating vs High Rating

In [19]:
# Mục tiêu: so sánh cân nặng và thể tích (volume) của sản phẩm
# giữa 2 nhóm: low rating (1–2 sao) và high rating (4–5 sao)
# Bỏ qua score = 3 (neutral) để tương phản rõ hơn
#
# product_volume_cm3 = length × height × width
# (đã được tính ở Feature Engineering trong notebook EDA)

rq2_q4 = """
    SELECT
        -- Phân nhóm rating
        CASE
            WHEN avg_review_score <= 2 THEN 'Low Rating (1–2 sao)'
            WHEN avg_review_score >= 4 THEN 'High Rating (4–5 sao)'
        END                                             AS rating_group,

        COUNT(*)                                        AS total_orders,

        -- Cân nặng sản phẩm (gram)
        ROUND(AVG(product_weight_g),  1)                AS avg_weight_g,
        ROUND(MIN(product_weight_g),  1)                AS min_weight_g,
        ROUND(MAX(product_weight_g),  1)                AS max_weight_g,

        -- Thể tích sản phẩm (cm³)
        -- = product_length_cm × product_height_cm × product_width_cm
        ROUND(
            AVG(
                product_length_cm * product_height_cm * product_width_cm
            ), 1
        )                                               AS avg_volume_cm3,

        -- Phí vận chuyển trung bình (sản phẩm nặng/to thường phí cao hơn)
        ROUND(AVG(total_freight), 2)                    AS avg_freight_cost

    FROM  orders
    WHERE avg_review_score   IS NOT NULL
      AND product_weight_g   IS NOT NULL
      AND avg_review_score   != 3          -- Bỏ qua neutral

    GROUP BY rating_group
    ORDER BY rating_group DESC
"""

rq2_result4 = run_query(rq2_q4,
    "RQ2-Q4: Cân Nặng & Thể Tích Sản Phẩm — Low Rating vs High Rating")
display(rq2_result4)

print("\n Diễn giải:")
print("  - Nếu avg_weight_g của Low Rating > High Rating")
print("    → sản phẩm nặng hơn có xu hướng bị đánh giá thấp hơn")
print("  - avg_freight_cost cũng phản ánh sự phức tạp vận chuyển")


 RQ2-Q4: Cân Nặng & Thể Tích Sản Phẩm — Low Rating vs High Rating


,rating_group,total_orders,avg_weight_g,min_weight_g,max_weight_g,avg_volume_cm3,avg_freight_cost
0,Low Rating (1–2 sao),13849,"2,314.0000",60.0000,"22,537.5000","16,492.9000",26.7600
1,High Rating (4–5 sao),75926,"2,020.6000",60.0000,"22,537.5000","14,661.6000",21.5200
2,None,59,"1,692.0000",100.0000,"16,800.0000","14,210.1000",23.5200



 Diễn giải:
  - Nếu avg_weight_g của Low Rating > High Rating
    → sản phẩm nặng hơn có xu hướng bị đánh giá thấp hơn
  - avg_freight_cost cũng phản ánh sự phức tạp vận chuyển


---
## 4. RQ3 — Geographic Factors & Logistics Efficiency

> **RQ3:** How does interstate shipping (seller and customer located in different states) affect delivery time, freight cost, and customer satisfaction?

**Chiến lược phân tích:**
- So sánh `same_state = 0` (interstate) vs `same_state = 1` (intrastate)
- Phân tích theo từng bang để thấy pattern địa lý
- Kết hợp cả 3 yếu tố: delivery time, freight, review score

### RQ3 — Query 5: Interstate vs Intrastate — Giao Hàng, Phí, và Satisfaction

In [20]:
# Mục tiêu: so sánh toàn diện giữa:
#   same_state = 1 → người bán và người mua cùng bang (intrastate)
#   same_state = 0 → khác bang (interstate)
# Kỳ vọng: interstate → giao hàng lâu hơn, phí cao hơn, review thấp hơn

rq3_q5 = """
    SELECT
        -- Nhãn rõ ràng cho 2 nhóm
        CASE same_state
            WHEN 1 THEN 'Cùng bang (Intrastate)'
            WHEN 0 THEN 'Khác bang (Interstate)'
        END                                             AS shipping_type,

        COUNT(*)                                        AS total_orders,

        -- ── Delivery metrics ────────────────────────────────
        -- Thời gian giao thực tế
        ROUND(AVG(actual_delivery_days),    1)          AS avg_delivery_days,

        -- Số ngày trễ so với dự kiến
        ROUND(AVG(delay_days),              1)          AS avg_delay_days,

        -- Tỉ lệ giao trễ
        ROUND(
            100.0 * SUM(is_late_delivery) / COUNT(*), 1
        )                                               AS late_delivery_pct,

        -- ── Cost metrics ─────────────────────────────────────
        -- Phí vận chuyển trung bình
        ROUND(AVG(total_freight),           2)          AS avg_freight_cost,

        -- ── Satisfaction metrics ─────────────────────────────
        -- Review score trung bình
        ROUND(AVG(avg_review_score),        2)          AS avg_review_score,

        -- Tỉ lệ đơn bị low rating (≤ 2 sao)
        ROUND(
            100.0 * SUM(CASE WHEN avg_review_score <= 2 THEN 1 ELSE 0 END)
                  / COUNT(*), 1
        )                                               AS low_rating_pct

    FROM  orders
    WHERE order_status = 'delivered'
      AND same_state   IS NOT NULL

    GROUP BY same_state
    ORDER BY same_state DESC     -- Cùng bang trước, khác bang sau
"""

rq3_result5 = run_query(rq3_q5,
    "RQ3-Q5: So Sánh Toàn Diện — Interstate vs Intrastate Shipping")
display(rq3_result5)

print("\n Diễn giải:")
print("  - avg_delivery_days: interstate nên cao hơn intrastate")
print("  - avg_freight_cost:  vận chuyển xa hơn → phí cao hơn")
print("  - avg_review_score:  nếu interstate thấp hơn → địa lý ảnh hưởng satisfaction")


 RQ3-Q5: So Sánh Toàn Diện — Interstate vs Intrastate Shipping


,shipping_type,total_orders,avg_delivery_days,avg_delay_days,late_delivery_pct,avg_freight_cost,avg_review_score,low_rating_pct
0,Cùng bang (Intrastate),34706,7.5000,-9.7000,5.4000,15.4500,4.2600,10.4000
1,Khác bang (Interstate),61772,14.7000,-12.1000,8.8000,26.2900,4.1000,14.0000



 Diễn giải:
  - avg_delivery_days: interstate nên cao hơn intrastate
  - avg_freight_cost:  vận chuyển xa hơn → phí cao hơn
  - avg_review_score:  nếu interstate thấp hơn → địa lý ảnh hưởng satisfaction


### RQ3 — Query Bonus: Top Bang Khách Hàng có Delivery Time Cao Nhất (Interstate)

In [21]:
# Mục tiêu: tìm bang nào của khách hàng bị ảnh hưởng nhiều nhất
# khi mua từ bang khác (interstate)
# Useful cho paper: minh hoạ sự bất bình đẳng địa lý trong logistics

rq3_bonus = """
    SELECT
        customer_state                                  AS customer_state,

        COUNT(*)                                        AS interstate_orders,

        -- Thời gian giao hàng trung bình đến bang này
        ROUND(AVG(actual_delivery_days), 1)             AS avg_delivery_days,

        -- Delay trung bình
        ROUND(AVG(delay_days), 1)                       AS avg_delay_days,

        -- Phí vận chuyển trung bình
        ROUND(AVG(total_freight), 2)                    AS avg_freight_cost,

        -- Review score trung bình của khách ở bang này
        ROUND(AVG(avg_review_score), 2)                 AS avg_review_score

    FROM  orders
    WHERE order_status          = 'delivered'
      AND same_state            = 0              -- Chỉ lấy interstate
      AND actual_delivery_days  IS NOT NULL

    GROUP BY customer_state

    -- Chỉ lấy bang có ít nhất 200 đơn interstate để đủ dữ liệu
    HAVING COUNT(*) >= 200

    ORDER BY avg_delivery_days DESC
    LIMIT 10
"""

rq3_bonus_result = run_query(rq3_bonus,
    "RQ3-Bonus: Top 10 Bang Khách Hàng có Thời Gian Giao Hàng Lâu Nhất (Interstate)")
display(rq3_bonus_result)

print("\n Diễn giải:")
print("  - Bang ở xa trung tâm logistics Brazil (SP, RJ) thường bị delay nhiều hơn")
print("  - avg_review_score thấp ở các bang này → địa lý → logistics → satisfaction")
print("  - Kết quả này có thể visualize bằng heatmap bản đồ Brazil ở bước tiếp theo")


 RQ3-Bonus: Top 10 Bang Khách Hàng có Thời Gian Giao Hàng Lâu Nhất (Interstate)


,customer_state,interstate_orders,avg_delivery_days,avg_delay_days,avg_freight_cost,avg_review_score
0,AL,397,24.0000,-8.2000,37.2700,3.8500
1,PA,946,23.3000,-13.5000,37.9900,3.9100
2,MA,703,21.3000,-9.0000,41.4100,3.8300
3,SE,335,21.0000,-9.5000,39.3900,3.9100
4,CE,1270,20.9000,-10.1000,34.9500,3.9400
5,PB,516,20.0000,-12.7000,43.8300,4.0700
6,RN,453,19.5000,-13.0000,39.4700,4.1100
7,PI,475,19.0000,-10.7000,41.0700,3.9900
8,BA,3190,19.0000,-10.2000,29.2800,3.9200
9,RO,243,18.9000,-19.5000,44.6600,4.1700



 Diễn giải:
  - Bang ở xa trung tâm logistics Brazil (SP, RJ) thường bị delay nhiều hơn
  - avg_review_score thấp ở các bang này → địa lý → logistics → satisfaction
  - Kết quả này có thể visualize bằng heatmap bản đồ Brazil ở bước tiếp theo


---
## 5. Tổng hợp kết quả & Export

In [22]:
# Export tất cả kết quả query ra CSV để dùng ở bước Visualization

results = {
    "rq1_delivery_by_score"       : rq1_result1,
    "rq1_late_vs_ontime"          : rq1_result2,
    "rq2_category_low_rating"     : rq2_result3,
    "rq2_product_physical"        : rq2_result4,
    "rq3_interstate_vs_intrastate": rq3_result5,
    "rq3_state_delivery_time"     : rq3_bonus_result,
}

os.makedirs("sql_results", exist_ok=True)

for filename, df_result in results.items():
    path = f"sql_results/{filename}.csv"
    df_result.to_csv(path, index=False)
    print(f"   {path}  ({df_result.shape[0]} rows × {df_result.shape[1]} cols)")

print("\n Export hoàn tất! Các file CSV trong thư mục sql_results/")
print("   Dùng làm input cho notebook Visualization tiếp theo.")

# Đóng kết nối SQLite
conn.close()
print("\n Đóng kết nối SQLite")

   sql_results/rq1_delivery_by_score.csv  (5 rows × 6 cols)
   sql_results/rq1_late_vs_ontime.csv  (2 rows × 6 cols)
   sql_results/rq2_category_low_rating.csv  (15 rows × 5 cols)
   sql_results/rq2_product_physical.csv  (3 rows × 7 cols)
   sql_results/rq3_interstate_vs_intrastate.csv  (2 rows × 8 cols)
   sql_results/rq3_state_delivery_time.csv  (10 rows × 6 cols)

 Export hoàn tất! Các file CSV trong thư mục sql_results/
   Dùng làm input cho notebook Visualization tiếp theo.

 Đóng kết nối SQLite


---
## 6. Tóm tắt Queries

| # | Query | RQ | Mục tiêu |
|---|-------|----|----------|
| Q1 | Delivery metrics theo Review Score (1–5) | RQ1 | Xem delivery time/delay/freight thay đổi theo rating |
| Q2 | Đơn trễ vs Đúng hạn → Review Score | RQ1 | Tác động trực tiếp của late delivery đến satisfaction |
| Q3 | Top danh mục có Low Rating cao nhất | RQ2 | Danh mục nguy cơ cao bị đánh giá xấu |
| Q4 | Cân nặng & thể tích: Low vs High Rating | RQ2 | Đặc điểm vật lý ảnh hưởng rating |
| Q5 | Interstate vs Intrastate: giao hàng, phí, review | RQ3 | So sánh toàn diện 2 loại hình vận chuyển |
| Bonus | Top bang có delivery time cao nhất (interstate) | RQ3 | Pattern địa lý → logistics inequality |

**Lưu ý cho paper (Methods section):**
- Tất cả queries đều lọc `order_status = 'delivered'` để đảm bảo có đủ dữ liệu delivery
- `avg_review_score ≤ 2` được dùng làm ngưỡng **"Low Rating"** (thang 1–5)
- `HAVING COUNT(*) >= 100/200` để đảm bảo ý nghĩa thống kê khi so sánh nhóm